In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("ml_challenge_dataset.csv")
df

In [ ]:
# Renaming Column names

df = df.rename(columns={'unique_id': 'rid',
                        'Painting': 'label', 'On a scale of 1–10, how intense is the emotion conveyed by the artwork?': 'intensity',
                        'Describe how this painting makes you feel.': 'feeling',
                        'This art piece makes me feel sombre.': 'sombre?',
                        'This art piece makes me feel content.': 'content?',
                        'This art piece makes me feel calm.': 'calm?',
                        'This art piece makes me feel uneasy.': 'uneasy?',
                        'How many prominent colours do you notice in this painting?': 'num_colors',
                        'How many objects caught your eye in the painting?': 'num_objects',
                        'How much (in Canadian dollars) would you be willing to pay for this painting?': 'worth',
                        'If you could purchase this painting, which room would you put that painting in?': 'place',
                        'If you could view this art in person, who would you want to view it with?': 'view_with',
                        'What season does this art piece remind you of?': 'season',
                        'If this painting was a food, what would be?': 'food',
                        'Imagine a soundtrack for this painting. Describe that soundtrack without naming any objects in the painting.': 'music'})

In [ ]:
# Aarushi

# replacing N/A in numerical columns with mean/median of the column.
df['intensity'] = df['intensity'].fillna(df['intensity'].median())
df['num_colors'] = df['num_colors'].fillna(df['num_colors'].median())
df['num_objects'] = df['num_objects'].fillna(df['num_objects'].median())

cols = ['sombre?', 'content?', 'calm?', 'uneasy?']
for col in cols:
    df[col] = df[col].str.extract(r'(\d+)').astype(float)

for col in cols:
    df[col] = df[col].fillna(df[col].median())

In [ ]:
# cleaning the feeling column and applying TF-IDF vectorization to it.

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

custom_stops = ['feel', 'feeling', 'feels', 'gives', 'makes', 
                'make', 'like', 'bit', 'sense', 'world', 
                'painting', 'little', 'life', 'away', 'look',
                'looking', 'really', 'reminds', 'think', 'way', 'wonder', 'want']

vectorizer = TfidfVectorizer(
    max_features=30, 
    stop_words=list(ENGLISH_STOP_WORDS) + custom_stops  # combine with built-in stops
)

feeling_tfidf = vectorizer.fit_transform(df['feeling'].fillna(''))

feeling_df = pd.DataFrame(feeling_tfidf.toarray(),
                          columns=vectorizer.get_feature_names_out())
df = pd.concat([df.reset_index(drop=True), feeling_df], axis=1)
df = df.drop(columns=['feeling']) # drop the old feelings column

In [ ]:
print(df.columns)
print(vectorizer.get_feature_names_out())
feeling_cols = [col for col in df.columns if col in vectorizer.get_feature_names_out()]
print(df[feeling_cols].iloc[1])

In [ ]:
# Adding column class

loc_index = df.columns.get_loc('label') + 1
new_col_values = []
for i in range(len(df)):
  j = i // 562
  new_col_values.append('{}'.format(j))
df.insert(loc=loc_index, column='Class', value=new_col_values)

In [ ]:
df.head(-50)